In [2]:
"""
Car Evaluation - Decision Tree Classifier
--------------------------------------------------------
Goal: predict a car's overall acceptability/safety rating
(unacc / acc / good / vgood) based on 6 categorical features.

Dataset: UCI Car Evaluation Data Set (1728 rows, no missing values,
ALL features are categorical -- this is a different flavor of
problem than Titanic, where we had numeric features like Age/Fare).

Steps:
1. Load the data (no header row in the raw file, so we add column names)
2. Encode all categorical text into numbers
3. Split into train/test
4. Train a Decision Tree
5. Score the model + look at what it learned
"""

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Load the data
#    The raw UCI file has no header row, so we supply column
#    names ourselves, in the order UCI documents them.
# ---------------------------------------------------------
col_names = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
# Load dataset from the UCI repository URL (avoids local path issues / typos)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
df = pd.read_csv(url, names=col_names)

print("Shape of dataset:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())   # UCI guarantees 0 missing values here

print("\nRaw data preview:")
print(df.head())

print("\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {sorted(df[col].unique())}")

# ---------------------------------------------------------
# 2. Encode categorical text into numbers
#
#    Every single column here is categorical text (no numeric
#    columns like Age/Fare this time). Importantly, most of these
#    categories have a natural ORDER (low < med < high < vhigh),
#    so instead of one-hot encoding we use OrdinalEncoder with an
#    explicit category order -- this preserves that "low is less
#    than high" relationship, which helps the tree split smartly.
# ---------------------------------------------------------
buying_order   = ["low", "med", "high", "vhigh"]
maint_order    = ["low", "med", "high", "vhigh"]
doors_order    = ["2", "3", "4", "5more"]
persons_order  = ["2", "4", "more"]
lug_boot_order = ["small", "med", "big"]
safety_order   = ["low", "med", "high"]
class_order    = ["unacc", "acc", "good", "vgood"]   # our target label order

encoder = OrdinalEncoder(categories=[
    buying_order, maint_order, doors_order,
    persons_order, lug_boot_order, safety_order
])

feature_cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]
X = pd.DataFrame(
    encoder.fit_transform(df[feature_cols]),
    columns=feature_cols
)

# Encode the target (class) the same ordinal way
class_map = {label: i for i, label in enumerate(class_order)}
y = df["class"].map(class_map)

print("\nEncoded features preview:")
print(X.head())
print("\nClass distribution (0=unacc, 1=acc, 2=good, 3=vgood):")
print(y.value_counts().sort_index())

# ---------------------------------------------------------
# 3. Train/test split
#    stratify=y keeps the same class proportions in both the
#    train and test sets -- important here since the classes
#    are imbalanced (most cars are "unacc").
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining rows: {len(X_train)}, Test rows: {len(X_test)}")

# ---------------------------------------------------------
# 4. Train the Decision Tree
# ---------------------------------------------------------
model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=6,
    random_state=42
)
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n=== Model Score ===")
print(f"Accuracy on test set: {accuracy:.3f}  ({accuracy*100:.1f}%)")

cv_scores = cross_val_score(model, X, y, cv=5)
print(f"5-fold Cross-Validation Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print("\nConfusion Matrix (rows = actual, columns = predicted):")
print("Order: unacc, acc, good, vgood")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_order))

# ---------------------------------------------------------
# 6. Feature importance -- which factor matters most for
#    a car's overall acceptability?
# ---------------------------------------------------------
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature Importance (higher = more useful for splitting):")
print(importances)

# ---------------------------------------------------------
# 7. Print the top of the tree so you can see the actual
#    questions it learned to ask
# ---------------------------------------------------------
print("\n=== Tree structure (top 3 levels) ===")
print(export_text(model, feature_names=feature_cols, max_depth=3))

Shape of dataset: (1728, 7)

Missing values per column:
buying      0
maint       0
doors       0
persons     0
lug_boot    0
safety      0
class       0
dtype: int64

Raw data preview:
  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc

Unique values per column:
  buying: ['high', 'low', 'med', 'vhigh']
  maint: ['high', 'low', 'med', 'vhigh']
  doors: ['2', '3', '4', '5more']
  persons: ['2', '4', 'more']
  lug_boot: ['big', 'med', 'small']
  safety: ['high', 'low', 'med']
  class: ['acc', 'good', 'unacc', 'vgood']

Encoded features preview:
   buying  maint  doors  persons  lug_boot  safety
0     3.0    3.0    0.0      0.0       0.0     0.0
1     3.0    3.0    0.0      0.0       0.0     1.0
2     3.0    3.0    0.0      0